## Market Regimes

We will attempt to identify market states using supervised learning. I often find when I'm trading that markets shift very frequently. One week I can only extract a max of 1 or 2 Rs from a scalp trade and then another week the market is giving 3R or 4R trades. 

In [54]:
# import libraries
import numpy as np
import pandas as pd
import joblib
from math import log2
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import GaussianMixture

In [55]:
# define csv file
rth_df = pd.read_csv("multiasset_5m_rth_features.csv")

print("Shape:", rth_df.shape)
print("Columns:")
print(rth_df.columns.tolist())

Shape: (171717, 27)
Columns:
['symbol', 'timestamp', 'ts_ny', 'date_ny', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'ret_1', 'ret_3', 'ret_6', 'ret_12', 'atr_14', 'atr_pct_14', 'vwap_session', 'vwap_dist', 'vwap_dist_atr', 'vol_ratio_20', 'range_pct', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'rv_12', 'tod_sin', 'tod_cos']


In [56]:
# ensure timestamps account for DST
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)
rth_df.head()

,symbol,timestamp,ts_ny,date_ny,open,high,low,close,volume,trade_count,...,vwap_dist,vwap_dist_atr,vol_ratio_20,range_pct,body_pct,upper_wick_pct,lower_wick_pct,rv_12,tod_sin,tod_cos
0,COIN,2024-02-28 18:35:00+00:00,2024-02-28 18:35:00+00:00,2024-02-28,201.26,202.930,201.1400,201.8700,210678.0,2749.0,...,-0.012738,-0.957690,0.531252,0.008867,0.003022,0.005251,0.000594,0.007565,-0.721202,-0.692724
1,COIN,2024-02-28 18:40:00+00:00,2024-02-28 18:40:00+00:00,2024-02-28,201.86,203.280,201.6800,202.1871,183109.0,2558.0,...,-0.010920,-0.827792,0.476845,0.007913,0.001618,0.005405,0.000890,0.007084,-0.774605,-0.632445
2,COIN,2024-02-28 18:45:00+00:00,2024-02-28 18:45:00+00:00,2024-02-28,202.30,203.820,201.8701,203.6350,280522.0,3099.0,...,-0.003521,-0.291903,0.726583,0.009575,0.006556,0.000908,0.002111,0.007378,-0.822984,-0.568065
3,COIN,2024-02-28 18:50:00+00:00,2024-02-28 18:50:00+00:00,2024-02-28,203.64,204.000,201.8602,203.6400,156113.0,2210.0,...,-0.003390,-0.289911,0.409781,0.010508,0.000000,0.001768,0.008740,0.007164,-0.866025,-0.500000
4,COIN,2024-02-28 18:55:00+00:00,2024-02-28 18:55:00+00:00,2024-02-28,203.77,205.585,203.3413,205.5189,232314.0,3074.0,...,0.005721,0.508484,0.614974,0.010917,0.008510,0.000322,0.002086,0.007683,-0.903450,-0.428693


In [57]:
# sort by stock and time stamp
rth_df = rth_df.sort_values(["ts_ny", "symbol"]).reset_index(drop=True)

print("Columns count:", len(rth_df.columns))
print("Symbols:", rth_df["symbol"].unique())
print("Date range:", rth_df["ts_ny"].min(), "to", rth_df["ts_ny"].max())

Columns count: 27
Symbols: <StringArray>
['TSLA', 'NVDA', 'PLTR', 'COIN', 'OKLO']
Length: 5, dtype: str
Date range: 2024-02-28 18:25:00+00:00 to 2026-02-27 16:55:00+00:00


In [58]:
# check for missing features
feature_cols = [
    "ret_1","ret_3","ret_6","ret_12",
    "atr_14","atr_pct_14",
    "vwap_session","vwap_dist","vwap_dist_atr",
    "vol_ratio_20",
    "range_pct","body_pct","upper_wick_pct","lower_wick_pct",
    "rv_12",
    "tod_sin","tod_cos"
]

missing_cols = [c for c in feature_cols if c not in rth_df.columns]
print("Missing feature columns:", missing_cols)

Missing feature columns: []


In [59]:
# validate price data is in chronological order
per_symbol_sorted = rth_df.groupby("symbol")["ts_ny"].apply(lambda s: s.is_monotonic_increasing)
print("\nEach Stock is in Chrono Order?")
print(per_symbol_sorted)


Each Stock is in Chrono Order?
symbol
COIN    True
NVDA    True
OKLO    True
PLTR    True
TSLA    True
Name: ts_ny, dtype: bool


In [60]:
# drop rows with NaNs features
mask_valid = rth_df[feature_cols].notna().all(axis=1)
print("Valid rows for clustering:", mask_valid.sum(), "/", len(rth_df))
print("Dropped rows due to NaNs:", (~mask_valid).sum())

Valid rows for clustering: 171717 / 171717
Dropped rows due to NaNs: 0


In [61]:
# verify price data range and daily trading sessions
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)

rth_df["ts_local"] = rth_df["ts_ny"].dt.tz_convert("America/New_York")
rth_df["session_date"] = rth_df["ts_local"].dt.date

print("Calendar date range:", rth_df["session_date"].min(), "->", rth_df["session_date"].max())
print("Unique trading sessions:", rth_df["session_date"].nunique())

Calendar date range: 2024-02-28 -> 2026-02-27
Unique trading sessions: 484


In [62]:
# calculate daily summary features

# ensure stocks are sorted by time stamp
rth_df = rth_df.sort_values(["symbol", "ts_ny"]).reset_index(drop=True)

# calculate return from close to close for each stock
rth_df["ret_cc"] = rth_df.groupby("symbol")["close"].pct_change()

# calculate daily price data for each stock
daily = (
    rth_df
    .groupby(["symbol", "session_date"])
    .agg(
        day_open=("open","first"),
        day_close=("close","last"),
        day_high=("high","max"),
        day_low=("low","min"),
        day_volume=("volume","sum"),
        bars_in_day=("close","size"),
        daily_rv=("ret_cc", lambda x: np.nanstd(x, ddof=0)),
    )
    .reset_index()
)

daily["daily_ret"] = daily["day_close"] / daily["day_open"] - 1.0
daily["daily_abs_ret"] = daily["daily_ret"].abs()
daily["daily_range_pct"] = (daily["day_high"] - daily["day_low"]) / daily["day_open"]
daily["daily_trend_strength"] = daily["daily_ret"].abs() / (daily["daily_range_pct"] + 1e-12)

print("Daily table shape:", daily.shape)
print("\nSample rows:")
print(daily.head())

Daily table shape: (2237, 13)

Sample rows:
  symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
0   COIN   2024-02-28    201.26    200.730   206.045  199.2400   4737289.0   
1   COIN   2024-02-29    206.46    203.420   211.310  193.8800  14412653.0   
2   COIN   2024-03-01    202.70    205.830   206.390  196.0101   8514752.0   
3   COIN   2024-03-04    217.40    228.720   236.460  212.2469  20685818.0   
4   COIN   2024-03-05    230.00    216.774   239.980  215.4000  21519305.0   

   bars_in_day  daily_rv  daily_ret  daily_abs_ret  daily_range_pct  \
0           29  0.004005  -0.002633       0.002633         0.033812   
1           78  0.007627  -0.014724       0.014724         0.084423   
2           78  0.003980   0.015442       0.015442         0.051208   
3           78  0.008353   0.052070       0.052070         0.111376   
4           78  0.007343  -0.057504       0.057504         0.106870   

   daily_trend_strength  
0              0.077884  
1       

### Unsupervised Clustering

We want to mimic how I would actually trade with regimes. I would expect a regime to persist until price action shows me otherwise. 

Conceptually, if the past few trading sessions were choppy, I'd expect the current session to be choppy. If the past several sessions were trending, I'd expect the current trading session to offer trend.

In [63]:
# Create rolling macro features
daily = daily.sort_values(["symbol","session_date"]).reset_index(drop=True)

# using 3 day rolling period
N = 3
LONG_N = 10
eps = 1e-8

# short term realized volatility
daily["rv_3"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term realized volatility
daily["rv_long"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# volatility expansion or contraction
daily["rv_impulse"] = np.log((daily["rv_3"] + eps) / (daily["rv_long"] + eps))

# range
daily["range_3"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term range
daily["range_long"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# range expansion or compression
daily["range_ratio"] = np.log((daily["range_3"] + eps) / (daily["range_long"] + eps))

# trend frequency
trend_thresh = 0.6
daily["trend_freq"] = (
    daily.groupby("symbol")["daily_trend_strength"]
    .transform(lambda s: (s.shift(1) > trend_thresh).rolling(N).mean())
)

# directional or chop
daily["sum_ret_3"] = (
    daily.groupby("symbol")["daily_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["sum_abs_ret_3"] = (
    daily.groupby("symbol")["daily_abs_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["conditions_3"] = daily["sum_ret_3"].abs() / (daily["sum_abs_ret_3"] + eps)

# relative volume vs past 10 days
daily["rel_vol"] = (
    daily.groupby("symbol")["day_volume"]
    .transform(lambda s: np.log(s.shift(1) / (s.shift(1).rolling(LONG_N).median() + eps)))
)

macro_feature_cols = [
    "rv_impulse",
    "range_ratio",
    "trend_freq",
    "conditions_3",
    "rel_vol"
]

print(daily[["symbol", "session_date"] + macro_feature_cols].head(15))

   symbol session_date  rv_impulse  range_ratio  trend_freq  conditions_3  \
0    COIN   2024-02-28         NaN          NaN         NaN           NaN   
1    COIN   2024-02-29         NaN          NaN         NaN           NaN   
2    COIN   2024-03-01         NaN          NaN    0.000000           NaN   
3    COIN   2024-03-04         NaN          NaN    0.000000      0.058424   
4    COIN   2024-03-05         NaN          NaN    0.000000      0.641898   
5    COIN   2024-03-06         NaN          NaN    0.000000      0.080047   
6    COIN   2024-03-07         NaN          NaN    0.000000      0.238468   
7    COIN   2024-03-08         NaN          NaN    0.000000      0.049591   
8    COIN   2024-03-11         NaN          NaN    0.000000      1.000000   
9    COIN   2024-03-12         NaN          NaN    0.333333      0.035393   
10   COIN   2024-03-13    0.043653     0.095391    0.333333      0.186174   
11   COIN   2024-03-14   -0.152740    -0.089752    0.333333      1.000000   

In [64]:
# normalize by stock looking back 30 days

norm_cols = macro_feature_cols.copy()
z_cols = []

Z_LOOKBACK = 30
eps = 1e-8

for col in norm_cols:
    zcol = f"{col}_z"
    daily[zcol] = (
        daily.groupby("symbol")[col]
        .transform(lambda s: (s - s.shift(1).rolling(Z_LOOKBACK).mean()) /
                             (s.shift(1).rolling(Z_LOOKBACK).std() + eps))
    )
    z_cols.append(zcol)

daily.tail(20)

,symbol,session_date,day_open,day_close,day_high,day_low,day_volume,bars_in_day,daily_rv,daily_ret,...,trend_freq,sum_ret_3,sum_abs_ret_3,conditions_3,rel_vol,rv_impulse_z,range_ratio_z,trend_freq_z,conditions_3_z,rel_vol_z
2217,TSLA,2026-01-22,435.190,449.3700,449.5000,432.6293,63597499.0,78,0.001825,0.032583,...,0.333333,-0.004924,0.051407,0.095786,0.112616,0.959168,0.929559,-0.059946,-1.670872,0.483165
2218,TSLA,2026-01-23,447.425,449.0700,452.4300,444.0400,51146282.0,78,0.001852,0.003677,...,0.666667,0.032278,0.079372,0.406673,0.176865,0.804153,0.999414,0.839248,-0.691032,0.683038
2219,TSLA,2026-01-26,445.020,435.2000,445.0500,434.2800,43782576.0,78,0.002705,-0.022066,...,0.333333,0.059502,0.059502,1.000000,-0.041020,0.074529,0.604429,-0.120567,0.929516,-0.150879
2220,TSLA,2026-01-27,437.410,430.9100,437.5200,430.6900,33965748.0,78,0.001275,-0.014860,...,0.666667,0.014194,0.058326,0.243348,-0.184723,0.019257,-0.051125,0.767744,-1.332384,-0.679170
2221,TSLA,2026-01-28,431.910,431.4600,438.2600,430.1000,42249391.0,78,0.001916,-0.001042,...,0.666667,-0.033250,0.040603,0.818901,-0.418231,-0.253718,-1.097607,0.714742,0.346854,-1.507316
2222,TSLA,2026-01-29,437.640,416.4400,440.2299,414.6200,70903529.0,78,0.002825,-0.048442,...,0.666667,-0.037969,0.037969,1.000000,-0.199994,-0.259478,-1.042554,0.663278,0.865322,-0.649539
2223,TSLA,2026-01-30,425.770,430.6200,439.8800,422.7000,74741961.0,78,0.003738,0.011391,...,0.666667,-0.064344,0.064344,1.000000,0.306076,-0.197647,0.377955,0.612892,0.835249,1.263686
2224,TSLA,2026-02-02,421.250,421.7850,427.1500,414.5000,51665167.0,78,0.004390,0.001270,...,0.333333,-0.038092,0.060875,0.625751,0.338332,1.020240,1.046097,-0.430620,-0.257366,1.455297
2225,TSLA,2026-02-03,424.270,421.8100,428.5600,413.6901,52032872.0,78,0.002471,-0.005798,...,0.333333,-0.035780,0.061103,0.585578,-0.015418,1.876163,1.416954,-0.430620,-0.339737,0.122000
2226,TSLA,2026-02-04,420.400,405.8300,423.9000,399.1800,67848231.0,78,0.003329,-0.034657,...,0.000000,0.006863,0.018459,0.371788,0.003540,1.812003,0.524653,-1.395197,-1.066138,0.268624


In [65]:
# drop NaNs
macro_z_cols = ["rv_impulse","range_ratio","trend_freq","conditions_3","rel_vol",
                "rv_impulse_z","range_ratio_z","trend_freq_z","conditions_3_z","rel_vol_z"]
macro = daily.dropna(subset=macro_z_cols).copy()

print(macro.head())

   symbol session_date  day_open  day_close  day_high  day_low  day_volume  \
40   COIN   2024-04-25    216.25     223.61    225.94   213.64   4451838.0   
41   COIN   2024-04-26    220.77     236.44    237.02   218.66   5472576.0   
42   COIN   2024-04-29    229.94     218.14    230.32   216.54   8487485.0   
43   COIN   2024-04-30    214.67     204.02    216.57   202.59   7891047.0   
44   COIN   2024-05-01    199.00     209.93    218.52   198.20   8957379.0   

    bars_in_day  daily_rv  daily_ret  ...  trend_freq  sum_ret_3  \
40           78  0.005675   0.034035  ...    1.000000   0.047609   
41           78  0.004153   0.070979  ...    0.666667   0.038390   
42           78  0.005703  -0.051318  ...    0.666667   0.052106   
43           78  0.004589  -0.049611  ...    0.666667   0.053696   
44           78  0.005568   0.054925  ...    1.000000  -0.029950   

    sum_abs_ret_3  conditions_3   rel_vol  rv_impulse_z  range_ratio_z  \
40       0.153423      0.310310 -0.359211     -1

In [66]:
entries = pd.read_csv("multiasset_labeled_entries.csv")
entries["ts_entry"] = pd.to_datetime(entries["ts_entry"], errors="coerce")
entries["ts_entry"] = entries["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")

min_val_ts  = entries.loc[entries["split"]=="val", "ts_entry"].min()
min_test_ts = entries.loc[entries["split"]=="test", "ts_entry"].min()

first_val_day  = min_val_ts.tz_convert("America/New_York").date()
first_test_day = min_test_ts.tz_convert("America/New_York").date()

print("first_val_day :", first_val_day)
print("first_test_day:", first_test_day)

first_val_day : 2025-07-17
first_test_day: 2025-10-29


In [67]:
# split macro data
macro_train = macro[macro["session_date"] < first_val_day].copy()
macro_val   = macro[(macro["session_date"] >= first_val_day) & (macro["session_date"] < first_test_day)].copy()
macro_test  = macro[macro["session_date"] >= first_test_day].copy()

print("Macro train rows:", len(macro_train))
print("Macro val rows  :", len(macro_val))
print("Macro test rows :", len(macro_test))

Macro train rows: 1359
Macro val rows  : 330
Macro test rows : 348


In [68]:
# scale macro features
macro_z_cols = ["rv_impulse_z","range_ratio_z","trend_freq_z","conditions_3_z","rel_vol_z"]
X_train_raw = macro_train[macro_z_cols].astype(float).values
X_val_raw   = macro_val[macro_z_cols].astype(float).values
X_test_raw  = macro_test[macro_z_cols].astype(float).values

# only fit training data
scaler_macro = RobustScaler()
X_train = scaler_macro.fit_transform(X_train_raw)
X_val   = scaler_macro.transform(X_val_raw)
X_test  = scaler_macro.transform(X_test_raw)

print(X_train.shape, X_val.shape, X_test.shape)

(1359, 5) (330, 5) (348, 5)


For clustering, we'll go with Gaussian Mixture Model.

https://scikit-learn.org/stable/modules/mixture.html

Stock regimes overlap so GMM will be better to use.

In [69]:
# GMM on training data
results = []
Ks = range(2, 6)
for K in Ks:
    gmm = GaussianMixture(
        n_components=K,
        covariance_type="diag",
        random_state=100,
        n_init=10,
        max_iter=800
    )
    gmm.fit(X_train)

    bic = gmm.bic(X_train)
    labels = gmm.predict(X_train)
    counts = np.bincount(labels, minlength=K)
    fracs = counts / counts.sum()

    results.append((K, bic, fracs.min(), fracs.max()))

    print(f"K={K}  BIC={bic:,.0f}  min_frac={fracs.min():.3f}  max_frac={fracs.max():.3f}")

C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Wi

K=2  BIC=14,508  min_frac=0.354  max_frac=0.646
K=3  BIC=14,174  min_frac=0.154  max_frac=0.530
K=4  BIC=14,107  min_frac=0.113  max_frac=0.353
K=5  BIC=13,942  min_frac=0.130  max_frac=0.290


C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Wi

Both K=4 and K=5 look strong, I'll go with K=5 for now.

In [70]:
# fit GMM
K = 4
gmm_macro = GaussianMixture(
    n_components=K,
    covariance_type="diag",
    random_state=100,
    n_init=20,
    max_iter=1000
)
gmm_macro.fit(X_train)

C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\miniconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Wi

,"n_components n_components: int, default=1The number of mixture components.",4
,"covariance_type covariance_type: {'full', 'tied', 'diag', 'spherical'}, default='full'String describing the type of covariance parameters to use.Must be one of:- 'full': each component has its own general covariance matrix.- 'tied': all components share the same general covariance matrix.- 'diag': each component has its own diagonal covariance matrix.- 'spherical': each component has its own single variance.For an example of using `covariance_type`, refer to:ref:`sphx_glr_auto_examples_mixture_plot_gmm_selection.py`.",'diag'
,"tol tol: float, default=1e-3The convergence threshold. EM iterations will stop when thelower bound average gain is below this threshold.",0.001
,"reg_covar reg_covar: float, default=1e-6Non-negative regularization added to the diagonal of covariance.Allows to assure that the covariance matrices are all positive.",1e-06
,"max_iter max_iter: int, default=100The number of EM iterations to perform.",1000
,"n_init n_init: int, default=1The number of initializations to perform. The best results are kept.",20
,"init_params init_params: {'kmeans', 'k-means++', 'random', 'random_from_data'}, default='kmeans'The method used to initialize the weights, the means and theprecisions.String must be one of:- 'kmeans' : responsibilities are initialized using kmeans.- 'k-means++' : use the k-means++ method to initialize.- 'random' : responsibilities are initialized randomly.- 'random_from_data' : initial means are randomly selected data points... versionchanged:: v1.1 `init_params` now accepts 'random_from_data' and 'k-means++' as initialization methods.",'kmeans'
,"weights_init weights_init: array-like of shape (n_components, ), default=NoneThe user-provided initial weights.If it is None, weights are initialized using the `init_params` method.",None
,"means_init means_init: array-like of shape (n_components, n_features), default=NoneThe user-provided initial means,If it is None, means are initialized using the `init_params` method.",None
,"precisions_init precisions_init: array-like, default=NoneThe user-provided initial precisions (inverse of the covariancematrices).If it is None, precisions are initialized using the 'init_params'method.The shape depends on 'covariance_type':: (n_components,) if 'spherical', (n_features, n_features) if 'tied', (n_components, n_features) if 'diag', (n_components, n_features, n_features) if 'full'",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to the method chosen to initialize theparameters (see `init_params`).In addition, it controls the generation of random samples from thefitted distribution (see the method `sample`).Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",100


In [71]:
# predict regimes for all data
X_all_raw = macro[macro_z_cols].astype(float).values
X_all = scaler_macro.transform(X_all_raw)

macro["macro_regime_id"] = gmm_macro.predict(X_all)

probs = gmm_macro.predict_proba(X_all)
macro["macro_regime_pmax"] = probs.max(axis=1)

print("Macro regime distribution:")
print(macro["macro_regime_id"].value_counts(normalize=True).sort_index())

print("\nMacro confidence summary:")
print(macro["macro_regime_pmax"].describe())

Macro regime distribution:
macro_regime_id
0    0.135493
1    0.111929
2    0.339224
3    0.413353
Name: proportion, dtype: float64

Macro confidence summary:
count    2037.000000
mean        0.791774
std         0.142371
min         0.348392
25%         0.684561
50%         0.822031
75%         0.905707
max         1.000000
Name: macro_regime_pmax, dtype: float64


In [72]:
pd.crosstab(macro["macro_regime_id"], macro["symbol"], normalize="index")

symbol,COIN,NVDA,OKLO,PLTR,TSLA
macro_regime_id,,,,,
0,0.228261,0.202899,0.163043,0.206522,0.199275
1,0.250000,0.197368,0.201754,0.188596,0.162281
2,0.205499,0.185239,0.205499,0.199711,0.204052
3,0.194774,0.194774,0.235154,0.186461,0.188836


In [73]:
macro.groupby("macro_regime_id")[macro_z_cols].mean()

,rv_impulse_z,range_ratio_z,trend_freq_z,conditions_3_z,rel_vol_z
macro_regime_id,,,,,
0,-1.287834,-1.573566,-0.741336,0.015084,-1.067065
1,1.782107,1.463654,0.425775,0.181929,1.556796
2,0.492639,0.681518,0.299369,-0.060065,0.471737
3,-0.374423,-0.423538,-0.089143,0.023543,-0.413789


In [74]:
# add regime to 5M price data
rth_df = rth_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "macro_regime_pmax"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Missing macro_regime_id on bars:", rth_df["macro_regime_id"].isna().sum())
print("Bars shape:", rth_df.shape)
print(rth_df[["symbol","ts_ny","session_date","macro_regime_id","macro_regime_pmax"]].head(10))

Missing macro_regime_id on bars: 14681
Bars shape: (171717, 32)
  symbol                     ts_ny session_date  macro_regime_id  \
0   COIN 2024-02-28 18:35:00+00:00   2024-02-28              NaN   
1   COIN 2024-02-28 18:40:00+00:00   2024-02-28              NaN   
2   COIN 2024-02-28 18:45:00+00:00   2024-02-28              NaN   
3   COIN 2024-02-28 18:50:00+00:00   2024-02-28              NaN   
4   COIN 2024-02-28 18:55:00+00:00   2024-02-28              NaN   
5   COIN 2024-02-28 19:00:00+00:00   2024-02-28              NaN   
6   COIN 2024-02-28 19:05:00+00:00   2024-02-28              NaN   
7   COIN 2024-02-28 19:10:00+00:00   2024-02-28              NaN   
8   COIN 2024-02-28 19:15:00+00:00   2024-02-28              NaN   
9   COIN 2024-02-28 19:20:00+00:00   2024-02-28              NaN   

   macro_regime_pmax  
0                NaN  
1                NaN  
2                NaN  
3                NaN  
4                NaN  
5                NaN  
6                NaN  
7  

In [75]:
# create regime dataset
rth_regime = rth_df.dropna(subset=["macro_regime_id"]).copy()
rth_regime["macro_regime_id"] = rth_regime["macro_regime_id"].astype(int)

print("Regime bars:", rth_regime.shape)
print("Dropped bars:", len(rth_df) - len(rth_regime))

Regime bars: (157036, 32)
Dropped bars: 14681


In [76]:
# add regimes to VWAP Reclaim Long labeled entries csv
entries_df = pd.read_csv("multiasset_labeled_entries.csv")
entries_df["ts_entry"] = pd.to_datetime(entries_df["ts_entry"], errors="coerce")
entries_df["ts_entry"] = entries_df["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")
entries_df["session_date"] = entries_df["ts_entry"].dt.tz_convert("America/New_York").dt.date

entries_df = entries_df.merge(
    macro[["symbol","session_date","macro_regime_id","macro_regime_pmax"]],
    on=["symbol","session_date"],
    how="left"
)

print("Entries:", len(entries_df))
print("Entries missing macro_regime_id:", entries_df["macro_regime_id"].isna().sum())

Entries: 1710
Entries missing macro_regime_id: 154


In [77]:
# create entries with regime dataset
entries_regime = entries_df.dropna(subset=["macro_regime_id"]).copy()
entries_regime["macro_regime_id"] = entries_regime["macro_regime_id"].astype(int)

print("Regime entries:", len(entries_regime))

Regime entries: 1556


In [79]:
# save new datasets
rth_regime.to_csv("multiasset_5m_rth_features_with_regime.csv", index=False)
entries_regime.to_csv("multiasset_labeled_entries_with_regime.csv", index=False)

Now to test Regimes on VWAP Reclaim Setups

In [80]:
opps = pd.read_csv("multiasset_labeled_entries_with_regime.csv")
opps["ts_entry"] = pd.to_datetime(opps["ts_entry"], errors="coerce")
opps["ts_entry"] = opps["ts_entry"].dt.tz_convert("UTC")

opps = opps.dropna(subset=["macro_regime_id"]).copy()
opps["macro_regime_id"] = opps["macro_regime_id"].astype(int)

print("Opps shape:", opps.shape)
print(opps["split"].value_counts())

Opps shape: (1556, 34)
split
train    1043
test      257
val       256
Name: count, dtype: int64


best_R is from the simulated results of the VWAP Reclaim setups.

best_R = maximum simulated R-multiple achievable for that setup.

If best_R > 0, at least one of stop/target/hold combination resulted in a profitable trade.

In [81]:
# Win = best_R > 0
# calculate metrics for each regime
opps["win"] = (opps["best_R"] > 0).astype(int)

def regime_perf(df):
    g = df.groupby("macro_regime_id").agg(
        n=("best_R","size"),
        win_rate=("win","mean"),
        avg_R=("best_R","mean"),
        med_R=("best_R","median"),
        std_R=("best_R","std"),
        p25_R=("best_R", lambda x: np.percentile(x, 25)),
        p75_R=("best_R", lambda x: np.percentile(x, 75)),
    )
    g["expectancy"] = g["avg_R"]
    return g.sort_index()

train_perf = regime_perf(opps[opps["split"]=="train"])
val_perf   = regime_perf(opps[opps["split"]=="val"])
test_perf  = regime_perf(opps[opps["split"]=="test"])

print("\nTrain:")
print(train_perf)

print("\nValidation:")
print(val_perf)

print("\nTest:")
print(test_perf)


Train:
                   n  win_rate     avg_R     med_R     std_R  p25_R     p75_R  \
macro_regime_id                                                                 
0                170  0.541176  1.027991  1.333333  2.003384   -1.0  2.666667   
1                106  0.650943  1.310087  1.786779  1.881909   -1.0  2.666667   
2                325  0.550769  1.043515  1.333333  2.008573   -1.0  2.666667   
3                442  0.540724  1.086062  1.033703  2.077537   -1.0  3.000000   

                 expectancy  
macro_regime_id              
0                  1.027991  
1                  1.310087  
2                  1.043515  
3                  1.086062  

Validation:
                   n  win_rate     avg_R     med_R     std_R  p25_R     p75_R  \
macro_regime_id                                                                 
0                 31  0.419355  0.739247 -1.000000  2.213463   -1.0  3.333333   
1                 20  0.650000  1.716667  2.000000  2.245919   -1.0  

In [82]:
opps.groupby(["macro_regime_id", "best_param_id"])["best_R"].mean().unstack()

best_param_id,S0.5_T0.75_H3,S0.5_T0.75_H6,S0.5_T1.0_H12,S0.5_T1.0_H3,S0.5_T1.0_H6,S0.5_T1.5_H12,S0.5_T1.5_H24,S0.5_T1.5_H3,S0.5_T1.5_H6,S0.5_T2.0_H12,...,S1.0_T1.0_H3,S1.0_T1.0_H6,S1.0_T1.5_H12,S1.0_T1.5_H24,S1.0_T1.5_H3,S1.0_T1.5_H6,S1.0_T2.0_H12,S1.0_T2.0_H24,S1.0_T2.0_H3,S1.0_T2.0_H6
macro_regime_id,,,,,,,,,,,,,,,,,,,,,
0,-0.861111,0.664320,NaN,2.0,NaN,NaN,NaN,2.813695,NaN,4.000000,...,1.0,NaN,NaN,1.099903,1.500000,NaN,2.0,2.0,2.000000,2.0
1,-0.634146,NaN,NaN,2.0,2.000000,NaN,NaN,2.943230,3.000000,4.000000,...,1.0,1.0,NaN,NaN,NaN,1.500000,2.0,NaN,1.857853,NaN
2,-0.707438,NaN,2.0,2.0,2.000000,3.0,NaN,2.937051,2.759029,4.000000,...,1.0,1.0,1.262809,NaN,1.500000,1.500000,2.0,2.0,2.000000,2.0
3,-0.668708,0.199294,2.0,2.0,1.593774,3.0,3.0,2.968776,3.000000,3.948182,...,1.0,1.0,1.500000,NaN,1.052159,1.258487,2.0,2.0,2.000000,2.0
